# 01 - Exploratory Data Analysis

## 1. Library Imports and Configuration


In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 


## 2. Continuous Dataset Loading

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/electricity-load-forecasting")

print("Path to dataset files:", path)

In [ ]:
df = pd.read_csv('../data/raw/continuous_dataset.csv', parse_dates=['datetime'], index_col='datetime')
df.sort_index()
df

## 3. Basic Dataset Inspection

In [ ]:
'''
Why this matters: inspect the first rows to understand each column,
confirm that the datetime field is parsed correctly, and check for
obvious anomalies such as missing values or negative demand.

Typical conclusion: the dataset contains hourly demand, weather, and
holiday features that can be used for supervised forecasting.
'''
df.head()


In [ ]:
df.info()
'''
Why this matters: review the number of rows, column types, and non-null
counts before modeling.

Typical conclusion: the dataset has about 50k hourly observations over
multiple years and `nat_demand` is usable as the forecasting target.
'''


In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(12,4))
df['nat_demand'].plot()
plt.title('National electricity demand - full series')
plt.ylabel('nat_demand')
plt.xlabel('Date')
plt.show()


In [ ]:
# Reconstruct one month
plt.figure(figsize=(12,4))
df['2018-01-01':'2018-01-31']['nat_demand'].plot()
plt.title('Electricity demand - January 2018')
plt.ylabel('nat_demand')
plt.xlabel('Date')
plt.show()


In [ ]:
# Reconstruct one day
plt.figure(figsize=(12, 6))
df['2018-01-09':'2018-01-09']['nat_demand'].plot()
plt.title('Electricity demand - one day')
plt.ylabel('nat_demand')
plt.xlabel('Date')
plt.show()


In [ ]:
# Reconstruct a typical day


In [ ]:
df['hour'] = df.index.hour
hourly_profile = df.groupby('hour')['nat_demand'].mean()

hourly_profile.plot(kind='line', marker='o', figsize=(12, 6))

plt.title('Average demand by hour of day')
plt.xlabel('Hour')
plt.ylabel('Average nat_demand')
plt.show()


In [ ]:
df['dayofweek'] = df.index.day_of_week
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

weekday_profile = df[df['is_weekend'] == 0].groupby('hour')['nat_demand'].mean()
weekend_profile = df[df['is_weekend'] == 1].groupby('hour')['nat_demand'].mean()
holiday_profile = df[df['holiday'] == 1].groupby('hour')['nat_demand'].mean()

plt.figure(figsize=(12,6))
weekday_profile.plot(marker='o', label='Weekday')
weekend_profile.plot(marker='o', label='Weekend')
holiday_profile.plot(marker='o', label='Holiday')

plt.title('Average demand by hour: weekday vs weekend vs holiday')
plt.xlabel('Hour')
plt.ylabel('Average nat_demand')
plt.legend()
plt.show()


In [ ]:
hourly_temperature_profile = df.groupby('hour')['T2M_toc'].mean()
hourly_demand_profile = df.groupby('hour')['nat_demand'].mean()

fig, ax1 = plt.subplots(figsize=(12,4))
ax1.plot(hourly_demand_profile, label='Demand', alpha=0.7)
ax1.set_ylabel('nat_demand')

ax2 = ax1.twinx()
ax2.plot(hourly_temperature_profile, label='T2M_toc (temperature)', alpha=0.7, linestyle='--')
ax2.set_ylabel('Temperature')

plt.title('Daily demand profile vs daily temperature profile')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Daily averages for demand and temperature
daily = df.resample('D').mean()

fig, ax1 = plt.subplots(figsize=(12,4))

ax1.plot(daily.index, daily['nat_demand'], label='Demand', alpha=0.7)
ax1.set_ylabel('nat_demand')

ax2 = ax1.twinx()
ax2.plot(daily.index, daily['T2M_toc'], label='T2M_toc (temperature)', alpha=0.7, linestyle='--', color='red')
ax2.set_ylabel('Temperature')

plt.title('Demand vs temperature - daily averages')
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(daily['T2M_toc'], daily['nat_demand'], alpha=0.3)
plt.xlabel('Average daily temperature (T2M_toc)')
plt.ylabel('Average daily demand (nat_demand)')
plt.title('Demand vs temperature relationship (daily)')
plt.show()


In [ ]:
df[['T2M_toc', 'T2M_san', 'T2M_dav']].describe()

In [ ]:
daily_temp = df[['T2M_toc', 'T2M_san', 'T2M_dav']].resample('D').mean()

daily_temp.plot(figsize=(12,4))
plt.title('Daily temperature across the 3 locations')
plt.ylabel('T2M (approx. deg C)')
plt.show()


In [ ]:
num_cols = [
    'nat_demand',
    'T2M_toc','QV2M_toc','TQL_toc','W2M_toc',
    'T2M_san','QV2M_san','TQL_san','W2M_san',
    'T2M_dav','QV2M_dav','TQL_dav','W2M_dav'
]

corr_with_nat = df[num_cols].corr()['nat_demand'].sort_values(ascending=False)
corr_with_nat


In [ ]:
df['Holiday_ID'].value_counts()


## Exploratory Data Analysis Summary

### 1. Dataset Structure

- **Source:** Electricity Load Forecasting (Kaggle).
- **Frequency:** hourly data.
- **Date range:** from `2015-01-03 01:00` to `2020-06-27 00:00` (about 5.5 years).
- **Size:** 48,048 rows and 16 numeric columns (`datetime` index).
- **Quality:** no missing values or duplicated timestamps were found.

Main columns:
- `nat_demand`: national electricity demand (target).
- Weather variables across 3 locations (`toc`, `san`, `dav`):
  `T2M_*` (temperature), `QV2M_*` (specific humidity),
  `TQL_*` (liquid water/cloud cover), `W2M_*` (wind speed).
- Calendar variables: `Holiday_ID`, `holiday` (0/1), `school` (0/1).

---

### 2. Demand Behavior (`nat_demand`)

Basic statistics:

- Mean around **1183**
- Standard deviation around **192**
- Minimum around **85**
- Maximum around **1755**

Interpretation:
- Hourly demand usually moves between roughly **1000 and 1400**.
- There are some unusually low outliers, but the overall range is plausible.

The full time series shows:
- A relatively stable series around the mean, without a strong long-term trend.
- Persistent oscillations plus a few isolated low points.

The monthly zoom, for example January 2018, clearly shows:
- **Daily seasonality:** demand falls overnight, rises during the day, and falls again at night.
- Differences between days that suggest a **weekly effect**.

The average hourly profile shows:
- Minimum demand around 0-5 h.
- A fast increase from 6-7 h.
- A main peak around 10-14 h.
- A second shoulder in the late afternoon and evening.

Conclusion: the model must capture a strong daily pattern.

---

### 3. Calendar Effects

Created variables:
- `hour`: hour of day (0-23).
- `dayofweek`: day of week (0=Monday, ..., 6=Sunday).
- `is_weekend`: 1 for Saturday or Sunday, otherwise 0.

Average hourly profiles were compared for:
- weekdays;
- weekends;
- holidays (`holiday=1`).

Results:
- The daily shape is similar across the three groups.
- The level changes: weekdays > weekends > holidays.
- The difference is especially large during central daytime hours, roughly 8-17 h.
- `Holiday_ID` has many infrequent categories, so the binary `holiday` flag is used first.
- `school` adds complementary calendar information.

Conclusion: calendar variables (`hour`, `dayofweek`, `is_weekend`, `holiday`, `school`) are highly relevant features.

---

### 4. Weather Effects

Temperature comparison:
- `T2M_toc`, `T2M_san`, and `T2M_dav` show very similar annual patterns.
- Approximate range: 20-39 deg C, with level differences across locations.

Correlation with `nat_demand`:

- `T2M_toc` around **0.65**
- `T2M_dav` around **0.65**
- `T2M_san` around **0.63**
- Wind (`W2M_*`): weak correlations, around 0.1-0.2.
- Cloud/liquid water (`TQL_*`) and humidity (`QV2M_*`): very low or near-zero correlations.

Daily demand vs temperature profile:
- Demand and temperature rise and fall at roughly similar times during the day.

Daily scatter plot:
- The relationship is positive but dispersed: temperature helps, but does not explain all demand variability.

Conclusion: **temperature is the most important weather variable**; wind, cloud cover, and humidity appear less informative for a first model.

---

### 5. Modeling Decisions from the EDA

For the first LSTM demand forecasting model:

1. **Target**
   - `nat_demand`.

2. **Main input features**
   - Historical `nat_demand` inside the lookback window.
   - Temperature at the 3 locations: `T2M_toc`, `T2M_san`, `T2M_dav`.
   - Calendar variables: `hour`, `dayofweek`, `is_weekend`, `holiday`, `school`.

3. **Rationale**
   - Daily and weekly demand patterns are strong, so calendar features are required.
   - Temperature is highly correlated with demand and has similar daily profiles, making it the key weather covariate.
   - Other weather variables show weaker correlations, so they can be excluded from the initial model to reduce noise.

This EDA supports the data preparation and LSTM windowing workflow and documents why these features were selected first.
